In [56]:
import pandas as pd
import numpy as np

In [57]:
df_train_path = r'../../IEEE-CIS-Fraud-Detection/Data\02_Processed/Clean_data_v2.csv'
df = pd.read_csv(df_train_path)

C:\Users\LỘC2007\AppData\Local\Temp\ipykernel_4404\3611518333.py:2: DtypeWarning: Columns (0: CNT_FAM_MEMBERS_GROUP) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(df_train_path)


In [58]:
df.head()

,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,ORGANIZATION_GROUP,NAME_INCOME_GROUP,NAME_HOUSING_GROUP,CNT_FAM_MEMBERS_GROUP,DEF_30_GROUP,DEF_60_GROUP,OBS_30_GROUP,OBS_60_GROUP,BUREAU_YEAR_GROUP,YEARS_BEGIN_GROUP
0,1,Cash loans,M,N,Y,0,202500.000,406597.5,24700.5,351000.0,...,Business Entity,Working,Owner,1,2-4,2,2-4,2-4,1,>=5
1,0,Cash loans,F,Y,Y,1,171000.000,1560726.0,41301.0,1395000.0,...,Business Entity,Commercial associate,Owner,3,0,0,1,1,2-3,>=5
2,0,Cash loans,F,N,Y,0,112500.000,1019610.0,33826.5,913500.0,...,Other/Unknown,Pensioner,Owner,2,0,0,1,1,1,>=5
3,0,Cash loans,F,N,Y,1,112500.000,652500.0,21177.0,652500.0,...,Utility/Essential,Working,Owner,3,0,0,0,0,0,>=5
4,0,Cash loans,F,N,Y,0,38419.155,148365.0,10678.5,135000.0,...,Other/Unknown,Pensioner,Owner,2,0,0,0,0,2-3,>=5


### Xử lý nhiễu dữ liệu & Chuyển đổi tuổi tác

- AGE: Đổi số ngày sinh âm thành số tuổi dương thực tế để mô hình họ Cây dễ tìm điểm cắt tối ưu (ví dụ: tuổi nhỏ hơn 25).

- Khử nhiễu: Đưa giá trị rác 365243 (1000 năm làm việc) về dạng khuyết (np.nan) để tránh làm lệch nhánh thuật toán.

In [59]:
# Khử rác hệ thống tự điền và tính số tuổi dương thực tế
df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
df["AGE"] = df["DAYS_BIRTH"].abs() / 365

C:\Users\LỘC2007\AppData\Local\Temp\ipykernel_4404\3956259614.py:2: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)


### Nhóm đặc trưng gánh nặng tài chính (Financial Burden)
- ANNUITY_TO_INCOME_RATIO: Tỷ lệ DTI (Nợ trên thu nhập). Áp lực trả nợ chiếm bao nhiêu % thu nhập tháng.

- CREDIT_TO_INCOME: Mức độ đòn bẩy tài chính (Vay gấp mấy lần thu nhập năm).

- PAYMENT_RATE: Đại diện cho thời hạn vay ngắn hay dài.

- INCOME_PER_PERSON: Thu nhập thực tế chia đều theo đầu người để thấy quỹ tiền dư thực tế.

In [60]:
# Tạo tỷ lệ gánh nặng nợ hàng tháng và đòn bẩy tài chính
df['ANNUITY_TO_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'] + 1e-5)
df['CREDIT_TO_INCOME'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1e-5)
df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'] + 1e-5)
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'] + 1e-5)

### Nhóm cấu trúc khoản vay & Tài sản mua (Deal Structure)
- LTV & GOODS_TO_CREDIT_RATIO: Tỷ lệ tài trợ của ngân hàng. Kiểm tra khách hàng có tự bỏ tiền túi ra trả trước (DOWN_PAYMENT) để cùng chịu rủi ro hay không.

- FLAG_OVER_BORROWING: Bắt bài hành vi vay số tiền lớn hơn cả giá trị món đồ để lấy tiền mặt tiêu xài việc khác (nhóm này có tỷ lệ bùng nợ rất cao). Dùng Vectorization (.astype(int)) để ép kiểu nhanh gấp 100 lần hàm .apply().

In [61]:
# Tính toán tỷ lệ tài trợ tài sản và phát hiện hành vi vay vượt giá trị đồ mua
df['GOODS_TO_CREDIT_RATIO'] = df['AMT_GOODS_PRICE'] / (df['AMT_CREDIT'] + 1e-5)
df['LTV'] = df["AMT_CREDIT"] / (df['AMT_GOODS_PRICE'] + 1e-5)
df['DOWN_PAYMENT'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']
df['FLAG_OVER_BORROWING'] = (df['AMT_CREDIT'] > df['AMT_GOODS_PRICE']).astype(int)

In [62]:
# 1. Tỷ lệ thời gian làm việc trên tuổi đời (Sử dụng abs vì DAYS_EMPLOYED và DAYS_BIRTH đang là số âm)
df['EMPLOYED_TO_AGE'] = abs(df['DAYS_EMPLOYED']) / abs(df['DAYS_BIRTH'])

# 2. Tỷ lệ tuổi thọ của xe so với tuổi của khách hàng (Đổi DAYS_BIRTH sang số năm dương)
df['CAR_TO_AGE'] = df['OWN_CAR_AGE'] / (df['DAYS_BIRTH'] / -365)

# 3. Tích của 3 nguồn điểm tín nhiệm ngoài
df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']

# 4. Trung bình cộng của 3 nguồn điểm
df['EXT_SOURCES_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)

# 5. Độ lệch chuẩn của 3 nguồn điểm (fillna bằng 0 phòng trường hợp chỉ có 1 nguồn điểm)
df['EXT_SOURCES_STD'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1).fillna(0)

In [63]:
df.to_csv('data_features_v2.csv',index=False)